# 🎯 Notebook 6 — Orchestrator Deployment and Routing Smoke Tests

The orchestrator is the single entry point for all Product Finder requests.  
It discovers available specialists at runtime, enforces governance constraints from metadata, computes dynamic execution order, and bundles the final answer.

## Dynamic orchestration flow (metadata-driven)
```
User query + governance context (persona, disclaimer_accepted)
        │
        ▼
[pf-orchestrator]
        │
        ├─ Step 0: discover_pf_agents()
        │    ├─ API Center MCP search "pf*" (mandatory first call)
        │    └─ hydrate and normalize governance metadata for each discovered agent
        │
        ├─ Step 1: choose contextualizer dynamically from metadata
        │    └─ derive intent, risk_tier, missing_context
        │
        ├─ Step 2: plan_dynamic_route(...)
        │    ├─ filter by metadata: personas, risk tiers, trust, verification, disclaimer
        │    ├─ apply dependency/order metadata (run_after, execution_order, stage)
        │    └─ produce route
        │
        ├─ Step 3: call specialists in planned order
        │    └─ aggregate outputs into final response
        │
        └─ Return governed bundle
```

## Bundle structure returned to caller
```json
{
  "final_answer": "...",
  "agents_used": ["..."],
  "routing_decision": {
    "intent": "...",
    "risk_tier": "...",
    "persona": "...",
    "disclaimer_accepted": false,
    "discovery_mode": "api_center_mcp_search"
  },
  "confidence": 0.91,
  "governance_notices": [],
  "disclaimer_required": false
}
```

In [ ]:
import sys, json, pathlib, time as _t, uuid, subprocess, datetime, os, re

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing shared/utils.py and workshop/product-finder")

repo_root = find_repo_root(pathlib.Path.cwd())
shared_dir = repo_root / "shared"
sys.path.insert(0, str(shared_dir))
import utils  # type: ignore

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(r.stderr or r.stdout).strip()}")
    return (r.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    r = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if r.returncode != 0:
        return default
    return (r.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    r = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(r.stderr or r.stdout).strip()}")

def next_version_tag(tag: str) -> str:
    match = re.fullmatch(r"v(\d+)", (tag or "").strip(), re.IGNORECASE)
    if not match:
        return "v1"
    return f"v{int(match.group(1)) + 1}"

def get_latest_v_tag_from_acr(repository: str) -> str:
    tags_out = run(
        f"az acr repository show-tags --name {ACR_NAME} --repository {repository} --output json",
        "",
        ""
    )
    if not tags_out.success:
        return ""

    tags = tags_out.json_data if isinstance(tags_out.json_data, list) else []
    latest = 0
    for tag in tags:
        if not isinstance(tag, str):
            continue
        match = re.fullmatch(r"v(\d+)", tag.strip(), re.IGNORECASE)
        if match:
            latest = max(latest, int(match.group(1)))

    return f"v{latest}" if latest > 0 else ""

# Runtime config from azd env.
SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
HUB_RG = azd_get("AZURE_RESOURCE_GROUP")
SPOKE_RG = azd_get("SPOKE_RESOURCE_GROUP")
FOUNDRY_ACCT = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
SPOKE_PROJECT = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
ACR_NAME = azd_get("SPOKE_ACR_NAME")
ACR_SERVER = azd_get("SPOKE_ACR_LOGIN_SERVER")
AZURE_LOCATION = azd_get("AZURE_LOCATION")

PF_MODEL_CONNECTION = azd_get_optional("PF_MODEL_CONNECTION", "Product-Finder-DEV-LLM")
PF_MODEL_DEPLOYMENT = azd_get_optional("PF_MODEL_DEPLOYMENT", "gpt-5.4-mini")
PF_MODEL_FULL = azd_get_optional("PF_MODEL_FULL", f"{PF_MODEL_CONNECTION}/{PF_MODEL_DEPLOYMENT}")
PF_MODEL_SUBSCRIPTION_KEY = azd_get_optional("PF_MODEL_SUBSCRIPTION_KEY", "")
ORCHESTRATOR_TAG_ENV_KEY = "PF_IMAGE_TAG_PF_ORCHESTRATOR"
PREVIOUS_IMAGE_TAG = get_latest_v_tag_from_acr("pf-orchestrator")
ORCHESTRATOR_IMAGE_TAG = PREVIOUS_IMAGE_TAG
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", f"https://{FOUNDRY_ACCT}.services.ai.azure.com/api/projects/{SPOKE_PROJECT}")

PF_DISCOVERY_MODE = azd_get_optional("PF_DISCOVERY_MODE", "api_center_live")
PF_API_CENTER_NAME = azd_get_optional("PF_API_CENTER_NAME", "")
PF_API_CENTER_WORKSPACE = azd_get_optional("PF_API_CENTER_WORKSPACE", "default")
PF_API_CENTER_API_VERSION = azd_get_optional("PF_API_CENTER_API_VERSION", "2024-06-01-preview")
PF_API_CENTER_APIS_URL = azd_get_optional("PF_API_CENTER_APIS_URL", "")
PF_API_CENTER_MCP_URL = azd_get_optional("PF_API_CENTER_MCP_URL", "")

APIM_GATEWAY_URL = azd_get_optional("APIM_GATEWAY_URL", "").rstrip("/")
PF_MODEL_AZURE_ENDPOINT = azd_get_optional("PF_MODEL_AZURE_ENDPOINT", APIM_GATEWAY_URL).rstrip("/")

# Specialist-call runtime config: default to APIM dynamic route.
PF_SPECIALIST_CALL_MODE = azd_get_optional("PF_SPECIALIST_CALL_MODE", "apim").strip().lower()
if not PF_SPECIALIST_CALL_MODE:
    PF_SPECIALIST_CALL_MODE = "apim"
if PF_SPECIALIST_CALL_MODE not in ("apim", "direct_foundry"):
    raise RuntimeError("PF_SPECIALIST_CALL_MODE must be 'apim' or 'direct_foundry'.")

PF_AGENT_ENDPOINT = azd_get_optional("PF_AGENT_ENDPOINT", "").rstrip("/")
if not PF_AGENT_ENDPOINT and APIM_GATEWAY_URL:
    PF_AGENT_ENDPOINT = f"{APIM_GATEWAY_URL}/agents"
PF_AGENT_SUBSCRIPTION_KEY = azd_get_optional("PF_AGENT_SUBSCRIPTION_KEY", "")

if not PF_API_CENTER_NAME:
    PF_API_CENTER_NAME = azd_get_optional("APIC_NAME", "")
if not PF_API_CENTER_NAME:
    raise RuntimeError("Missing API Center name. Run Notebook 5 first.")

if not PF_API_CENTER_APIS_URL:
    PF_API_CENTER_APIS_URL = (
        f"https://management.azure.com/subscriptions/{SUB_ID}"
        f"/resourceGroups/{HUB_RG}/providers/Microsoft.ApiCenter/services/{PF_API_CENTER_NAME}"
        f"/workspaces/{PF_API_CENTER_WORKSPACE}/apis?api-version={PF_API_CENTER_API_VERSION}"
    )

ORCHESTRATOR_NAME = "pf-orchestrator"
AGENTS_BASE = pathlib.Path("../agents").resolve()

utils.print_info(f"Orchestrator:         {ORCHESTRATOR_NAME}")
utils.print_info(f"Model:                {PF_MODEL_FULL}")
utils.print_info(f"Current image tag:    {PREVIOUS_IMAGE_TAG or '<none>'}")
utils.print_info(f"Next image tag:       {next_version_tag(PREVIOUS_IMAGE_TAG)}")
utils.print_info(f"Discovery mode:       {PF_DISCOVERY_MODE}")
utils.print_info(f"API Center service:   {PF_API_CENTER_NAME}")
utils.print_info(f"API Center workspace: {PF_API_CENTER_WORKSPACE}")
utils.print_info(f"API Center MCP URL:   {PF_API_CENTER_MCP_URL or '<not configured>'}")
utils.print_info(f"Specialist call mode: {PF_SPECIALIST_CALL_MODE}")

if APIM_GATEWAY_URL:
    utils.print_info(f"APIM gateway:         {APIM_GATEWAY_URL}")
if PF_SPECIALIST_CALL_MODE == "apim":
    utils.print_info(f"Specialist endpoint:  {PF_AGENT_ENDPOINT}")

if not PF_MODEL_AZURE_ENDPOINT:
    raise RuntimeError("Missing PF_MODEL_AZURE_ENDPOINT/APIM_GATEWAY_URL. Re-run Notebook 3 or set azd env.")
if not PF_MODEL_SUBSCRIPTION_KEY:
    raise RuntimeError("Missing PF_MODEL_SUBSCRIPTION_KEY. Re-run Notebook 1/3 to populate model access contract secrets.")
if PF_SPECIALIST_CALL_MODE == "apim":
    if not PF_AGENT_ENDPOINT:
        raise RuntimeError("Missing PF_AGENT_ENDPOINT/APIM_GATEWAY_URL for APIM specialist routing.")
    if not PF_AGENT_SUBSCRIPTION_KEY:
        raise RuntimeError("Missing PF_AGENT_SUBSCRIPTION_KEY for APIM specialist routing.")

utils.print_info(f"Foundry endpoint:     {FOUNDRY_EP}")
utils.print_info(f"Model endpoint:       {PF_MODEL_AZURE_ENDPOINT}")
utils.print_info(f"Model deployment:     {PF_MODEL_DEPLOYMENT}")
utils.print_info(f"ACR:                  {ACR_SERVER}")

In [ ]:
# ── 1️⃣  Write orchestrator source files ─────────────────────────────────────
if "AGENTS_BASE" not in globals() or "ORCHESTRATOR_NAME" not in globals():
    raise RuntimeError("Run Cell 2 first to load runtime configuration (AGENTS_BASE/ORCHESTRATOR_NAME).")
orch_dir = AGENTS_BASE / ORCHESTRATOR_NAME
orch_dir.mkdir(exist_ok=True)

ORCHESTRATOR_MAIN = '''import os, asyncio, json, uuid, requests, re
from collections import defaultdict
from typing import Annotated
from pydantic import Field
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential

_DISCOVERY_MODE = os.environ.get("PF_DISCOVERY_MODE", "api_center_live")
_API_CENTER_MCP_URL = os.environ.get("PF_API_CENTER_MCP_URL", "").strip().rstrip("/")
_API_CENTER_APIS_URL = os.environ["PF_API_CENTER_APIS_URL"]
_FOUNDRY_PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")
_FOUNDRY_AGENT_API_VERSION = os.environ.get("PF_FOUNDRY_AGENT_API_VERSION", "2025-05-15-preview")
_SPECIALIST_CALL_MODE = os.environ.get("PF_SPECIALIST_CALL_MODE", "apim").strip().lower()
_APIM_SPECIALIST_ENDPOINT = (os.environ.get("PF_AGENT_ENDPOINT") or os.environ.get("APIM_GATEWAY_URL") or "").strip().rstrip("/")
_APIM_SUB_KEY = (os.environ.get("PF_AGENT_SUBSCRIPTION_KEY") or "").strip()

_TOKEN_CRED = DefaultAzureCredential()


def _arm_token() -> str:
    token = _TOKEN_CRED.get_token("https://management.azure.com/.default")
    return token.token


def _foundry_token() -> str:
    token = _TOKEN_CRED.get_token("https://ai.azure.com/.default")
    return token.token


def _coerce_list(value: object) -> list[str]:
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        return [x.strip() for x in value.split(",") if x.strip()]
    return []


def _coerce_bool(value: object, default: bool = False) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        v = value.strip().lower()
        if v in ("true", "1", "yes", "y", "on"):
            return True
        if v in ("false", "0", "no", "n", "off"):
            return False
    if isinstance(value, (int, float)):
        return bool(value)
    return default


def _coerce_int(value: object, default: int) -> int:
    try:
        return int(str(value).strip())
    except Exception:
        return default


def _extract_output_text(payload: object) -> str:
    if isinstance(payload, dict):
        if isinstance(payload.get("output_text"), str):
            return payload["output_text"]
        output = payload.get("output")
        if isinstance(output, list):
            chunks = []
            for item in output:
                if not isinstance(item, dict):
                    continue
                for c in item.get("content", []):
                    if isinstance(c, dict) and c.get("type") in ("output_text", "text") and c.get("text"):
                        chunks.append(c["text"])
            if chunks:
                return "\\n".join(chunks)
        return json.dumps(payload)
    if isinstance(payload, str):
        return payload
    return str(payload)


def _route_suffix(agent_name: str) -> str:
    return re.sub(r"[^a-z0-9-]", "-", (agent_name or "").strip().lower())


def _call_specialist(agent_name: str, message: str, persona: str) -> str:
    try:
        payload = {
            "input": f"[GOVERNANCE CONTEXT]\\npersona: {persona}\\n\\n{message}",
            "metadata": {"conversation_id": str(uuid.uuid4()), "target_agent": agent_name},
        }

        if _SPECIALIST_CALL_MODE == "apim":
            if not _APIM_SPECIALIST_ENDPOINT or not _APIM_SUB_KEY:
                return json.dumps({
                    "error": "APIM specialist routing is enabled but PF_AGENT_ENDPOINT/PF_AGENT_SUBSCRIPTION_KEY are missing",
                    "agent": agent_name,
                })
            route = _route_suffix(agent_name)
            base = _APIM_SPECIALIST_ENDPOINT
            if base.lower().endswith("/agents"):
                url = f"{base}/{route}/invoke"
            else:
                url = f"{base}/agents/{route}/invoke"
            headers = {
                "Ocp-Apim-Subscription-Key": _APIM_SUB_KEY,
                "Content-Type": "application/json",
            }
        else:
            url = f"{_FOUNDRY_PROJECT_ENDPOINT}/agents/{agent_name}/endpoint/protocols/openai/responses?api-version={_FOUNDRY_AGENT_API_VERSION}"
            headers = {
                "Authorization": f"Bearer {_foundry_token()}",
                "Content-Type": "application/json",
            }

        last_status = None
        last_body = ""
        last_error = None

        for _ in range(3):
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=90)
                last_status = resp.status_code
                last_body = resp.text
                if resp.status_code < 500:
                    break
            except Exception as e:
                last_error = str(e)
                last_status = None
                continue

        if last_status is None or last_status >= 400:
            mode = "APIM" if _SPECIALIST_CALL_MODE == "apim" else "Direct Foundry"
            error_msg = f"{mode} specialist call failed: HTTP {last_status}"
            return json.dumps({
                "error": error_msg,
                "agent": agent_name,
                "body": (last_body or "")[:1200],
                "exception": last_error,
            })
        try:
            return _extract_output_text(resp.json())
        except Exception:
            return resp.text
    except Exception as e:
        return json.dumps({"error": str(e), "agent": agent_name})


def _parse_governance_profile(api_item: dict) -> dict:
    props = api_item.get("properties") if isinstance(api_item.get("properties"), dict) else {}
    custom = props.get("customProperties") if isinstance(props.get("customProperties"), dict) else {}

    profile_raw = custom.get("governanceProfile")
    profile = {}
    if isinstance(profile_raw, str) and profile_raw.strip():
        try:
            profile = json.loads(profile_raw)
        except Exception:
            profile = {}

    # Flattened metadata fallback (legacy + explicit fields).
    profile.setdefault("agent_name", custom.get("agentName") or props.get("title") or api_item.get("name") or "")
    profile.setdefault("description", custom.get("description") or custom.get("summary") or props.get("description") or "")
    profile.setdefault("capabilities", _coerce_list(profile.get("capabilities") or custom.get("capabilities") or custom.get("skills") or ""))
    profile.setdefault("supported_intents", _coerce_list(profile.get("supported_intents") or custom.get("supportedIntents") or "*"))
    profile.setdefault("allowed_personas", _coerce_list(profile.get("allowed_personas") or custom.get("allowedPersonas") or "*"))
    profile.setdefault("risk_tiers_supported", _coerce_list(profile.get("risk_tiers_supported") or custom.get("riskTiersSupported") or "low,elevated"))
    profile.setdefault("run_after", _coerce_list(profile.get("run_after") or custom.get("runAfter") or custom.get("dependsOn") or ""))
    profile.setdefault("orchestration_stage", str(profile.get("orchestration_stage") or custom.get("orchestrationStage") or "execution").strip().lower())
    profile.setdefault("execution_order", _coerce_int(profile.get("execution_order") or custom.get("executionOrder"), 500))
    profile.setdefault("priority", _coerce_int(profile.get("priority") or custom.get("priority"), 999))
    profile.setdefault("trust_level", str(profile.get("trust_level") or custom.get("trustLevel") or "unknown").strip().lower())
    profile.setdefault("verification_status", str(profile.get("verification_status") or custom.get("verificationStatus") or "unknown").strip().lower())
    profile.setdefault("requires_disclaimer", _coerce_bool(profile.get("requires_disclaimer") if "requires_disclaimer" in profile else custom.get("requiresDisclaimer"), False))
    profile.setdefault("auth_required", _coerce_bool(profile.get("auth_required") if "auth_required" in profile else custom.get("authRequired"), False))
    profile.setdefault("enabled", _coerce_bool(profile.get("enabled") if "enabled" in profile else custom.get("enabled"), True))

    return profile


def _discover_from_api_center_rest() -> list[dict]:
    headers = {
        "Authorization": f"Bearer {_arm_token()}",
        "Content-Type": "application/json",
    }
    resp = requests.get(_API_CENTER_APIS_URL, headers=headers, timeout=45)
    resp.raise_for_status()
    body = resp.json()
    items = body.get("value", []) if isinstance(body, dict) else []
    profiles = []
    for item in items:
        profile = _parse_governance_profile(item)
        agent_name = str(profile.get("agent_name", "")).strip().lower()
        if agent_name.startswith("pf"):
            profiles.append(profile)
    return profiles


def _extract_mcp_assets(payload: object) -> list[dict]:
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if isinstance(payload, dict):
        for key in ("assets", "value", "items", "results", "data"):
            candidate = payload.get(key)
            if isinstance(candidate, list):
                return [x for x in candidate if isinstance(x, dict)]
        result = payload.get("result")
        if isinstance(result, dict):
            for key in ("assets", "value", "items", "results", "data"):
                candidate = result.get(key)
                if isinstance(candidate, list):
                    return [x for x in candidate if isinstance(x, dict)]
            content = result.get("content")
            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict):
                        text = block.get("text")
                        if not isinstance(text, str) or not text.strip():
                            continue
                        try:
                            parsed = json.loads(text)
                        except Exception:
                            continue
                        extracted = _extract_mcp_assets(parsed)
                        if extracted:
                            return extracted
    return []


def _mcp_search_pf_assets() -> list[dict]:
    # Always start discovery with MCP search for pf* assets.
    if not _API_CENTER_MCP_URL:
        return []

    url = f"{_API_CENTER_MCP_URL}/search"
    payloads = [
        {"query": "pf*"},
        {"search": "pf*"},
        {"namePrefix": "pf"},
    ]

    for body in payloads:
        try:
            resp = requests.post(url, json=body, timeout=25)
            if resp.status_code >= 400:
                continue
            assets = _extract_mcp_assets(resp.json())
            if assets:
                filtered = []
                for asset in assets:
                    name = str(
                        asset.get("name")
                        or asset.get("title")
                        or ((asset.get("properties") or {}).get("title") if isinstance(asset.get("properties"), dict) else "")
                    ).strip().lower()
                    if name.startswith("pf"):
                        filtered.append(asset)
                if filtered:
                    return filtered
        except Exception:
            continue
    return []


def _mcp_fetch_asset(asset_stub: dict) -> dict | None:
    asset_id = str(asset_stub.get("id") or asset_stub.get("name") or "").strip()
    if not asset_id:
        return None

    fetch_url = f"{_API_CENTER_MCP_URL}/assets/{asset_id}"
    try:
        resp = requests.get(fetch_url, timeout=25)
        if resp.status_code < 400:
            payload = resp.json()
            if isinstance(payload, dict):
                return payload
    except Exception:
        pass

    try:
        resp = requests.post(f"{_API_CENTER_MCP_URL}/fetch", json={"id": asset_id}, timeout=25)
        if resp.status_code < 400:
            payload = resp.json()
            if isinstance(payload, dict):
                return payload
    except Exception:
        pass

    return None


def _profile_from_mcp_asset(asset: dict) -> dict:
    props = asset.get("properties") if isinstance(asset.get("properties"), dict) else {}
    custom = props.get("customProperties") if isinstance(props.get("customProperties"), dict) else {}

    wrapped = {
        "name": asset.get("name"),
        "properties": {
            "title": props.get("title") or asset.get("title"),
            "description": props.get("description") or asset.get("description"),
            "customProperties": custom,
        },
    }
    profile = _parse_governance_profile(wrapped)
    if not profile.get("agent_name"):
        profile["agent_name"] = asset.get("name") or asset.get("title") or ""
    return profile


def _discover_profiles() -> dict:
    stubs = _mcp_search_pf_assets()
    if stubs:
        hydrated = []
        for stub in stubs:
            full = _mcp_fetch_asset(stub)
            hydrated.append(full if isinstance(full, dict) else stub)

        profiles = []
        for asset in hydrated:
            if not isinstance(asset, dict):
                continue
            profile = _profile_from_mcp_asset(asset)
            agent_name = str(profile.get("agent_name", "")).strip().lower()
            if agent_name.startswith("pf"):
                profiles.append(profile)

        if profiles:
            return {
                "discovery_mode": "api_center_mcp_search",
                "discovered_count": len(profiles),
                "profiles": profiles,
            }

    profiles = _discover_from_api_center_rest()
    return {
        "discovery_mode": "api_center_rest_fallback",
        "discovered_count": len(profiles),
        "profiles": profiles,
    }


def _matches_rule(value: str, allowed: list[str]) -> bool:
    if not allowed:
        return True
    normalized = {x.strip().lower() for x in allowed if x.strip()}
    return "*" in normalized or "any" in normalized or value.strip().lower() in normalized


def _is_contextualizer(profile: dict) -> bool:
    stage = str(profile.get("orchestration_stage", "")).lower()
    caps = {x.lower() for x in _coerce_list(profile.get("capabilities", []))}
    name = str(profile.get("agent_name", "")).lower()
    return (
        "context" in stage
        or "contextualize" in stage
        or "intent" in caps
        or "contextualization" in caps
        or name.endswith("contextualizer")
    )


def _agent_allowed(profile: dict, intent: str, persona: str, risk_tier: str, disclaimer_accepted: bool) -> tuple[bool, str]:
    if not _coerce_bool(profile.get("enabled"), True):
        return False, "disabled"

    if not _matches_rule(intent, _coerce_list(profile.get("supported_intents", []))):
        return False, "intent_not_supported"
    if not _matches_rule(persona, _coerce_list(profile.get("allowed_personas", []))):
        return False, "persona_not_allowed"
    if not _matches_rule(risk_tier, _coerce_list(profile.get("risk_tiers_supported", []))):
        return False, "risk_tier_not_supported"

    if _coerce_bool(profile.get("requires_disclaimer"), False) and risk_tier == "elevated" and not disclaimer_accepted:
        return False, "disclaimer_required"

    # For elevated risk, trust/verification can constrain usage.
    if risk_tier == "elevated":
        trust = str(profile.get("trust_level", "unknown")).lower()
        verification = str(profile.get("verification_status", "unknown")).lower()
        allow_unverified = _coerce_bool(profile.get("allow_elevated_unverified"), False)
        if not allow_unverified and trust not in ("high", "verified") and verification not in ("verified", "high"):
            return False, "trust_or_verification_too_low"

    return True, "allowed"


def _stage_rank(profile: dict) -> int:
    stage = str(profile.get("orchestration_stage", "")).strip().lower()
    explicit = _coerce_int(profile.get("execution_order"), 500)
    stage_defaults = {
        "contextualization": 100,
        "context": 100,
        "discovery": 200,
        "retrieval": 300,
        "analysis": 400,
        "reasoning": 500,
        "validation": 700,
        "alignment": 800,
        "fulfillment": 900,
    }
    return min(explicit, stage_defaults.get(stage, explicit))


def _topological_sort(profiles: list[dict]) -> list[dict]:
    by_name = {str(p.get("agent_name", "")).strip(): p for p in profiles if str(p.get("agent_name", "")).strip()}
    indegree = {name: 0 for name in by_name}
    graph = defaultdict(list)

    for name, profile in by_name.items():
        for dep in _coerce_list(profile.get("run_after", [])):
            dep_name = dep.strip()
            if dep_name in by_name and dep_name != name:
                graph[dep_name].append(name)
                indegree[name] += 1

    ready = [n for n, deg in indegree.items() if deg == 0]
    ready.sort(key=lambda n: (_stage_rank(by_name[n]), _coerce_int(by_name[n].get("priority"), 999), n))

    ordered_names = []
    while ready:
        current = ready.pop(0)
        ordered_names.append(current)
        for nxt in graph[current]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                ready.append(nxt)
        ready.sort(key=lambda n: (_stage_rank(by_name[n]), _coerce_int(by_name[n].get("priority"), 999), n))

    if len(ordered_names) != len(by_name):
        # Cycle fallback: deterministic stage+priority sort.
        ordered_names = sorted(
            by_name.keys(),
            key=lambda n: (_stage_rank(by_name[n]), _coerce_int(by_name[n].get("priority"), 999), n),
        )

    return [by_name[n] for n in ordered_names]


@tool(approval_mode="never_require")
def discover_pf_agents() -> str:
    """Always start orchestration by searching API Center MCP for pf* assets and returning governance profiles."""
    try:
        result = _discover_profiles()
        return json.dumps(result)
    except Exception as e:
        return json.dumps({
            "error": str(e),
            "discovery_mode": _DISCOVERY_MODE,
            "discovered_count": 0,
            "profiles": [],
        })


@tool(approval_mode="never_require")
def plan_dynamic_route(
    profiles_json: Annotated[str, Field(description="JSON array of discovered profiles")],
    intent: Annotated[str, Field(description="Detected intent")],
    persona: Annotated[str, Field(description="Request persona")],
    risk_tier: Annotated[str, Field(description="low|elevated")],
    disclaimer_accepted: Annotated[bool, Field(description="Whether risk disclaimer was accepted")] = True,
) -> str:
    """Build a metadata-driven specialist execution plan with deterministic governance filtering and ordering."""
    try:
        parsed = json.loads(profiles_json)
        profiles = parsed if isinstance(parsed, list) else []

        allowed = []
        blocked = []
        for p in profiles:
            if not isinstance(p, dict):
                continue
            ok, reason = _agent_allowed(
                profile=p,
                intent=intent,
                persona=persona,
                risk_tier=risk_tier,
                disclaimer_accepted=disclaimer_accepted,
            )
            if ok:
                allowed.append(p)
            else:
                blocked.append({"agent_name": p.get("agent_name"), "reason": reason})

        ordered = _topological_sort(allowed)
        route = [str(x.get("agent_name")) for x in ordered if str(x.get("agent_name", "")).strip()]

        blocked_reasons = [str(b.get("reason", "")) for b in blocked if isinstance(b, dict)]
        has_blocked_disclaimer = any(r == "disclaimer_required" for r in blocked_reasons)
        intent_norm = str(intent or "").strip().lower()
        risk_norm = str(risk_tier or "").strip().lower()
        disclaimer_gate_triggered = (
            (not disclaimer_accepted)
            and has_blocked_disclaimer
            and (intent_norm == "compatibility" or risk_norm == "elevated")
        )

        if disclaimer_gate_triggered:
            # Hard-stop route when disclaimer is mandatory but not accepted yet.
            ordered = []
            route = []

        disclaimer_required = (
            disclaimer_gate_triggered
            or any(_coerce_bool(x.get("requires_disclaimer"), False) for x in ordered)
            or has_blocked_disclaimer
        )

        # Check if core specialist for this intent was blocked (not a wrapper agent).
        core_specialist_blocked = False
        blocked_names = {str(b.get("agent_name", "")).strip().lower() for b in blocked}
        allowed_names = {str(a.get("agent_name", "")).strip().lower() for a in allowed}
        wrappers = {"pf-contextualizer", "pf-aligner"}
        core_specialists = {name for name in (blocked_names | allowed_names) if name not in wrappers and name.startswith("pf")}
        
        # If all non-wrapper specialists for this intent are blocked, flag it.
        intent_specific_specialists = core_specialists
        if intent_specific_specialists and all(name in blocked_names for name in intent_specific_specialists):
            core_specialist_blocked = True

        return json.dumps({
            "intent": intent,
            "persona": persona,
            "risk_tier": risk_tier,
            "allowed_count": len(route),
            "route": route,
            "ordered_profiles": ordered,
            "blocked": blocked,
            "disclaimer_gate_triggered": disclaimer_gate_triggered,
            "disclaimer_required": disclaimer_required,
            "core_specialist_blocked": core_specialist_blocked,
        })
    except Exception as e:
        return json.dumps({
            "error": str(e),
            "route": [],
            "ordered_profiles": [],
            "blocked": [],
            "disclaimer_gate_triggered": False,
            "disclaimer_required": False,
            "core_specialist_blocked": False,
        })


@tool(approval_mode="never_require")
def call_specialist_agent(
    agent_name: Annotated[str, Field(description="Target specialist agent name")],
    message: Annotated[str, Field(description="Message sent to specialist")],
    persona: Annotated[str, Field(description="Request persona")],
) -> str:
    """Call a selected specialist via APIM dynamic route (or direct Foundry in fallback mode)."""
    return _call_specialist(agent_name=agent_name, message=message, persona=persona)


ORCHESTRATOR_SYSTEM = """You are the Product Finder Orchestrator.
Your routing must be metadata-driven and dynamic.

You receive messages in this format:
[GOVERNANCE CONTEXT]
persona: external_customer|internal_scientist
disclaimer_accepted: true|false

USER QUERY: <the user's question>

Mandatory process:

STEP 0 (MANDATORY): DISCOVER AGENTS FIRST
- Always call discover_pf_agents before any other operation.
- This tool must be the first tool call in each conversation.
- Discover all pf* specialists first, then make routing decisions from their metadata.
- Use discovered profile metadata to decide routing; do not assume static workflows.

STEP 1: IDENTIFY A CONTEXTUALIZER DYNAMICALLY
- From discovered profiles, choose the contextualizer agent using metadata:
  - Prefer orchestration_stage indicating contextualization/context.
  - If multiple candidates, prefer lower execution_order then lower priority.
- Call that contextualizer with the full incoming message.
- Parse JSON for intent, risk_tier, and missing_context.

STEP 1A: HANDLE MISSING CONTEXT
- If missing_context is non-empty: stop and ask those clarifying questions.
- Do not call downstream specialists.
- Return final JSON with agents_used containing only the contextualizer.
- If missing_context is empty: continue immediately to STEP 2 in the same turn.

STEP 2: BUILD ROUTE FROM METADATA
- Call plan_dynamic_route with discovered profiles + intent + persona + risk_tier + disclaimer_accepted.
- Route selection must be driven by metadata filters (supported_intents, allowed_personas, risk_tiers_supported, trust/verification, disclaimer, enabled, dependencies/order).
- Use route exactly as returned; do not invent ad-hoc steps.
- If disclaimer_gate_triggered is true, stop and ask the user to accept the disclaimer first.
- When disclaimer_gate_triggered is true, do NOT call any downstream specialist and do NOT provide a compatibility verdict yet.
- **CRITICAL: If core_specialist_blocked is true (required specialist for this intent is not available due to governance constraints), immediately stop and return:**
  ```json
  {
    "final_answer": "I cannot execute this request based on your current role and governance constraints. The required capabilities are restricted by your access level.",
    "agents_used": [],
    "routing_decision": {
      "intent": "...",
      "risk_tier": "...",
      "persona": "...",
      "disclaimer_accepted": false,
      "discovery_mode": "..."
    },
    "confidence": 0.0,
    "governance_notices": ["Request blocked: Required specialist capabilities are not available for this role."],
    "disclaimer_required": false
  }
  ```
  Do NOT proceed to STEP 3 when core_specialist_blocked is true.
- If route is empty (but core_specialist_blocked is false), continue to STEP 3 normally with wrapper agents only.

STEP 3: EXECUTE ROUTE
- Call specialists in route order using call_specialist_agent.
- Never execute STEP 3 when disclaimer_gate_triggered is true.
- Skip re-calling the contextualizer if it already ran in STEP 1.
- Consolidate outputs into final_answer.

Governance rules:
- Persona/risk/disclaimer/trust constraints are enforced by metadata filtering.
- Elevated-risk requests must not bypass metadata eligibility.
- disclaimer_required is true whenever selected or blocked specialists indicate a disclaimer gate.

Return strict JSON:
{
  "final_answer": "well-formatted response for the user",
  "agents_used": ["list of specialist agent names called"],
  "routing_decision": {
    "intent": "...",
    "risk_tier": "...",
    "persona": "...",
    "disclaimer_accepted": false,
    "discovery_mode": "..."
  },
  "confidence": 0.0,
  "governance_notices": ["policy notices, disclaimers, or gating reasons"],
  "disclaimer_required": false
}
"""


async def setup():
    client = OpenAIChatClient(
        model=os.environ["PF_MODEL_DEPLOYMENT"],
        base_url=f"{os.environ['PF_MODEL_AZURE_ENDPOINT'].rstrip('/')}/models",
        api_key=os.environ["PF_MODEL_SUBSCRIPTION_KEY"],
        default_headers={"api-key": os.environ["PF_MODEL_SUBSCRIPTION_KEY"]},
    )
    agent = Agent(
        client=client,
        name="pf-orchestrator",
        instructions=ORCHESTRATOR_SYSTEM,
        tools=[discover_pf_agents, plan_dynamic_route, call_specialist_agent],
        default_options={"store": False},
    )
    return ResponsesHostServer(agent)


if __name__ == "__main__":
    asyncio.run(setup()).run()
'''

ORCH_REQUIREMENTS = """\
agent-framework>=1.2.0
agent-framework-foundry-hosting>=1.0.0a260507
azure-identity>=1.25.0
azure-monitor-opentelemetry>=1.6.0
requests>=2.32.0
"""

ORCH_DOCKERFILE = """\
FROM python:3.13-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY main.py .
EXPOSE 8088
CMD ["python", "main.py"]
"""

(orch_dir / "main.py").write_text(ORCHESTRATOR_MAIN.strip(), encoding="utf-8")
(orch_dir / "requirements.txt").write_text(ORCH_REQUIREMENTS, encoding="utf-8")
(orch_dir / "Dockerfile").write_text(ORCH_DOCKERFILE, encoding="utf-8")
(orch_dir / "agent.yaml").write_text(
    f"kind: hosted\nname: {ORCHESTRATOR_NAME}\nprotocols:\n"
    f"  - protocol: responses\n    version: 1.0.0\n"
    f"resources:\n  cpu: \"2\"\n  memory: 4Gi\n"
    f"environment_variables:\n"
    f"  - name: PF_MODEL_DEPLOYMENT\n    value: {PF_MODEL_DEPLOYMENT}\n"
    f"  - name: PF_MODEL_AZURE_ENDPOINT\n    value: {PF_MODEL_AZURE_ENDPOINT}\n"
    f"  - name: PF_MODEL_SUBSCRIPTION_KEY\n    value: ${{PF_MODEL_SUBSCRIPTION_KEY}}\n"
    f"  - name: FOUNDRY_PROJECT_ENDPOINT\n    value: {FOUNDRY_EP}\n"
    f"  - name: PF_FOUNDRY_AGENT_API_VERSION\n    value: 2025-05-15-preview\n"
    f"  - name: PF_SPECIALIST_CALL_MODE\n    value: {PF_SPECIALIST_CALL_MODE}\n"
    f"  - name: PF_AGENT_ENDPOINT\n    value: {PF_AGENT_ENDPOINT}\n"
    f"  - name: PF_AGENT_SUBSCRIPTION_KEY\n    value: ${{PF_AGENT_SUBSCRIPTION_KEY}}\n"
    f"  - name: PF_DISCOVERY_MODE\n    value: {PF_DISCOVERY_MODE}\n"
    f"  - name: PF_API_CENTER_MCP_URL\n    value: {PF_API_CENTER_MCP_URL}\n"
    f"  - name: PF_API_CENTER_APIS_URL\n    value: {PF_API_CENTER_APIS_URL}\n"
    f"  - name: OTEL_SERVICE_NAME\n    value: {ORCHESTRATOR_NAME}\n"
    f"  - name: ENABLE_INSTRUMENTATION\n    value: \"true\"\n",
    encoding="utf-8"
)
utils.print_ok(f"Orchestrator source files written to: {orch_dir.name}/")

In [ ]:
import time as _t

# ── 2️⃣  Build orchestrator image in ACR ─────────────────────────────────────
# Increment tag here, just before building, so deploy always uses the new image.
PREVIOUS_IMAGE_TAG = ORCHESTRATOR_IMAGE_TAG
ORCHESTRATOR_IMAGE_TAG = next_version_tag(PREVIOUS_IMAGE_TAG)
set_azd_env(ORCHESTRATOR_TAG_ENV_KEY, ORCHESTRATOR_IMAGE_TAG)
utils.print_info(f"{ORCHESTRATOR_NAME}: {PREVIOUS_IMAGE_TAG or '<none>'} -> {ORCHESTRATOR_IMAGE_TAG}")

utils.print_info(f"Building orchestrator image in ACR: {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG}")
build_out = run(
    f"az acr build --registry {ACR_NAME} --resource-group {SPOKE_RG} "
    f"--image {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} "
    f"--file {AGENTS_BASE / ORCHESTRATOR_NAME}/Dockerfile "
    f"--no-logs {AGENTS_BASE / ORCHESTRATOR_NAME}/",
    "Orchestrator build queued", "Orchestrator build failed"
)
if not build_out.success:
    # Roll back tag on build failure.
    set_azd_env(ORCHESTRATOR_TAG_ENV_KEY, PREVIOUS_IMAGE_TAG)
    ORCHESTRATOR_IMAGE_TAG = PREVIOUS_IMAGE_TAG
    raise RuntimeError("Failed to queue ACR build for orchestrator")

utils.print_info("Polling ACR for orchestrator image presence...")
for attempt in range(30):
    tag_out = run(
        f"az acr repository show-tags --name {ACR_NAME} "
        f"--repository {ORCHESTRATOR_NAME} --output json",
        "", ""
    )
    if tag_out.success and tag_out.json_data and ORCHESTRATOR_IMAGE_TAG in tag_out.json_data:
        utils.print_ok(f"{ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} verified in ACR")
        break
    utils.print_info(f"  Waiting for image... ({(attempt+1)*10}s)")
    _t.sleep(10)
else:
    raise RuntimeError(f"Image {ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG} not found in ACR after 5 min")

### 4️⃣ Deploy orchestrator to Foundry and wait for active status

Create a new orchestrator agent version from the freshly built image and poll until it becomes `active`.

In [ ]:
# ── 4️⃣  Deploy orchestrator to Foundry and wait for active status ───────────
import time as _t
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import HostedAgentDefinition
from azure.identity import DefaultAzureCredential

if "project_client" not in globals() or project_client is None:
    project_client = AIProjectClient(endpoint=FOUNDRY_EP, credential=DefaultAzureCredential(), allow_preview=True)
    utils.print_info("Initialized Foundry project client for this kernel session.")

image = f"{ACR_SERVER}/{ORCHESTRATOR_NAME}:{ORCHESTRATOR_IMAGE_TAG}"
utils.print_info(f"Deploying {ORCHESTRATOR_NAME} -> {image}")

orchestrator_version = project_client.agents.create_version(
    agent_name=ORCHESTRATOR_NAME,
    definition=HostedAgentDefinition({
        "container_protocol_versions": [{"protocol": "responses", "version": "1.0.0"}],
        "image": image,
        "cpu": "2",
        "memory": "4Gi",
        "environment_variables": {
            "PF_MODEL_DEPLOYMENT": PF_MODEL_DEPLOYMENT,
            "PF_MODEL_AZURE_ENDPOINT": PF_MODEL_AZURE_ENDPOINT,
            "PF_MODEL_SUBSCRIPTION_KEY": PF_MODEL_SUBSCRIPTION_KEY,
            "PF_FOUNDRY_AGENT_API_VERSION": "2025-05-15-preview",
            "PF_SPECIALIST_CALL_MODE": PF_SPECIALIST_CALL_MODE,
            "PF_AGENT_ENDPOINT": PF_AGENT_ENDPOINT,
            "PF_AGENT_SUBSCRIPTION_KEY": PF_AGENT_SUBSCRIPTION_KEY,
            "PF_DISCOVERY_MODE": PF_DISCOVERY_MODE,
            "PF_API_CENTER_MCP_URL": PF_API_CENTER_MCP_URL,
            "PF_API_CENTER_APIS_URL": PF_API_CENTER_APIS_URL,
            "OTEL_SERVICE_NAME": ORCHESTRATOR_NAME,
            "ENABLE_INSTRUMENTATION": "true",
            "ENABLE_SENSITIVE_DATA": "true",
        },
    }),
)

utils.print_ok(
    f"Created orchestrator version {orchestrator_version.version} "
    f"(status: {orchestrator_version.status})"
)

utils.print_info("Waiting for orchestrator to become active...")
MAX_WAIT = 600
POLL = 15
elapsed = 0

while elapsed < MAX_WAIT:
    v = project_client.agents.get_version(
        agent_name=orchestrator_version.name,
        agent_version=orchestrator_version.version,
    )
    status_raw = str(v.status or "")
    status_norm = status_raw.strip().lower()
    utils.print_info(f"  [{elapsed:>3}s] {ORCHESTRATOR_NAME}: {v.status}")

    if status_norm == "active" or status_norm.endswith(".active"):
        orchestrator_active_version = v
        utils.print_ok(f"{ORCHESTRATOR_NAME} is ACTIVE")
        break
    if status_norm in ("failed", "error") or status_norm.endswith(".failed") or status_norm.endswith(".error"):
        raise RuntimeError(f"{ORCHESTRATOR_NAME} deployment failed: {v.status}")

    _t.sleep(POLL)
    elapsed += POLL
else:
    raise RuntimeError(f"Timed out waiting for {ORCHESTRATOR_NAME} to become active")

### 5️⃣ Assign Foundry User RBAC to orchestrator identity

Resolve the orchestrator managed identity and assign `Foundry User` at the Foundry account scope. Wait for propagation when a new assignment is created.

In [ ]:
# ── 5️⃣  Assign Foundry User RBAC to orchestrator identity ───────────────────
foundry_scope = (
    f"/subscriptions/{SUB_ID}/resourceGroups/{SPOKE_RG}"
    f"/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_ACCT}"
)

identity = None
if "orchestrator_active_version" in globals() and orchestrator_active_version is not None:
    identity = orchestrator_active_version.instance_identity
if not identity:
    detail = project_client.agents.get(agent_name=ORCHESTRATOR_NAME)
    identity = detail.instance_identity

if not identity:
    utils.print_warning(
        f"{ORCHESTRATOR_NAME}: could not resolve managed identity; assign Foundry User manually"
    )
else:
    pid = identity.principal_id
    check = subprocess.run(
        [
            "az", "role", "assignment", "list",
            "--assignee-object-id", pid,
            "--role", "Foundry User",
            "--scope", foundry_scope,
            "--query", "[0].id",
            "-o", "tsv",
        ],
        capture_output=True,
        text=True,
        shell=True,
    )
    if check.stdout.strip():
        utils.print_ok(f"{ORCHESTRATOR_NAME}: Foundry User role already assigned")
    else:
        result = subprocess.run(
            [
                "az", "role", "assignment", "create",
                "--assignee-object-id", pid,
                "--assignee-principal-type", "ServicePrincipal",
                "--role", "Foundry User",
                "--scope", foundry_scope,
                "--only-show-errors",
                "-o", "none",
            ],
            capture_output=True,
            text=True,
            shell=True,
        )
        if result.returncode == 0:
            utils.print_ok(
                f"{ORCHESTRATOR_NAME}: Foundry User assigned (principal: {pid[:8]}...)"
            )
            utils.print_info("Waiting 60s for RBAC propagation...")
            _t.sleep(60)
            utils.print_ok("RBAC propagation wait complete.")
        else:
            utils.print_warning(
                f"{ORCHESTRATOR_NAME}: RBAC assignment failed - {result.stderr.strip()}"
            )

    # API Center Reader role is required for discovery via ARM URL.
    api_center_scope = (
        f"/subscriptions/{SUB_ID}/resourceGroups/{HUB_RG}"
        f"/providers/Microsoft.ApiCenter/services/{PF_API_CENTER_NAME}"
    )
    reader_check = subprocess.run(
        [
            "az", "role", "assignment", "list",
            "--assignee-object-id", pid,
            "--scope", api_center_scope,
            "-o", "json",
        ],
        capture_output=True,
        text=True,
        shell=True,
    )
    has_reader = False
    if reader_check.returncode == 0:
        try:
            assignments = json.loads(reader_check.stdout or "[]")
            has_reader = any((a.get("roleDefinitionName") == "Reader") for a in assignments)
        except Exception:
            has_reader = False

    if has_reader:
        utils.print_ok(f"{ORCHESTRATOR_NAME}: API Center Reader role already assigned")
    else:
        reader_create = subprocess.run(
            [
                "az", "role", "assignment", "create",
                "--assignee-object-id", pid,
                "--assignee-principal-type", "ServicePrincipal",
                "--role", "Reader",
                "--scope", api_center_scope,
                "--only-show-errors",
                "-o", "none",
            ],
            capture_output=True,
            text=True,
            shell=True,
        )
        if reader_create.returncode == 0:
            utils.print_ok(f"{ORCHESTRATOR_NAME}: API Center Reader role assigned")
            utils.print_info("Waiting 30s for API Center RBAC propagation...")
            _t.sleep(30)
            utils.print_ok("API Center RBAC propagation wait complete.")
        else:
            utils.print_warning(
                f"{ORCHESTRATOR_NAME}: API Center Reader assignment failed - {reader_create.stderr.strip()}"
            )